# Retail Intelligence Platform - SQL Reviews and Delivery Analysis

This notebook builds the **reviews and delivery analytics layer** for the Retail Intelligence Platform using PostgreSQL.

## Objectives
- analyze review score distribution and review quality
- calculate average review score and rating mix
- evaluate delivery performance and late delivery behavior
- measure delivery delays and on-time delivery rate
- analyze review score by delivery performance
- export clean reviews and delivery outputs to `sql/outputs/`

## PostgreSQL tables used
- `olist_orders`
- `olist_order_reviews`
- `olist_order_items`

## Reusable view used
- `vw_sales_fact`

In [28]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

# --------------------------------------------------
# PostgreSQL connection
# --------------------------------------------------
DB_USER = "postgres"
DB_PASSWORD = quote_plus("1234")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "retail_intelligence"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("PostgreSQL engine created successfully.")

PostgreSQL engine created successfully.


## 1. Validate review and order tables
We first confirm that review and order records exist and the database connection is working correctly.


In [29]:
validation_query = """
SELECT
    (SELECT COUNT(*) FROM olist_order_reviews) AS total_reviews,
    (SELECT COUNT(*) FROM olist_orders) AS total_orders;
"""

df_validation = pd.read_sql(validation_query, engine)
df_validation

,total_reviews,total_orders
0,99224,99441


## 2. Create reviews and delivery analytical view

This section creates a reusable view combining:
- order dates
- estimated and actual delivery dates
- delivery delay metrics
- review scores
- order status

In [30]:
create_reviews_delivery_view_sql = """
DROP VIEW IF EXISTS vw_reviews_delivery_summary;

CREATE VIEW vw_reviews_delivery_summary AS
WITH delivered_orders AS (
    SELECT
        o.order_id,
        o.order_purchase_timestamp::date AS order_purchase_date,
        o.order_delivered_customer_date::date AS delivered_date,
        o.order_estimated_delivery_date::date AS estimated_date
    FROM olist_orders o
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp IS NOT NULL
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
),

review_agg AS (
    SELECT
        r.order_id,
        AVG(r.review_score) AS review_score
    FROM olist_order_reviews r
    GROUP BY r.order_id
)

SELECT
    d.order_id,
    (d.delivered_date - d.order_purchase_date) AS delivery_days,
    (d.estimated_date - d.order_purchase_date) AS estimated_delivery_days,
    (d.delivered_date - d.estimated_date) AS delivery_variance_days,
    CASE
        WHEN d.delivered_date > d.estimated_date THEN 1
        ELSE 0
    END AS is_late,
    CASE
        WHEN d.delivered_date > d.estimated_date
        THEN (d.delivered_date - d.estimated_date)
        ELSE NULL
    END AS late_delay_days,
    r.review_score
FROM delivered_orders d
LEFT JOIN review_agg r
    ON d.order_id = r.order_id;
"""

with engine.begin() as conn:
    conn.exec_driver_sql(create_reviews_delivery_view_sql)

print("vw_reviews_delivery_summary created successfully.")

vw_reviews_delivery_summary created successfully.


## 3. Preview reviews and delivery view

In [31]:
reviews_delivery_preview_query = """
SELECT *
FROM vw_reviews_delivery
LIMIT 10;
"""

df_reviews_delivery_preview = pd.read_sql(reviews_delivery_preview_query, engine)
df_reviews_delivery_preview

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_creation_date,review_answer_timestamp,delivery_delay_days,delivery_lead_days,delivery_status
0,73fc7af87114b39712e6da79b0a377eb,delivered,2018-01-11 15:30:49,2018-01-11 15:47:59,2018-01-12 21:57:22,2018-01-17 18:42:41,2018-02-02,7bc2406110b926393aa56f80a40eba40,4,2018-01-18,2018-01-18 21:46:59,-16,6,On Time
1,a548910a1c6147796b98fdf73dbeba33,delivered,2018-02-28 12:25:19,2018-02-28 12:48:39,2018-03-02 19:08:15,2018-03-09 23:17:20,2018-03-14,80e641a11e56f04c1ad469d5645fdfde,5,2018-03-10,2018-03-11 03:05:13,-5,9,On Time
2,f9e4b658b201a9f2ecdecbb34bed034b,delivered,2018-02-03 09:56:22,2018-02-03 10:33:41,2018-02-06 16:18:28,2018-02-16 17:28:48,2018-03-09,228ce5500dc1d8e020d8d1322874b6f0,5,2018-02-17,2018-02-18 14:36:24,-21,13,On Time
3,658677c97b385a9be170737859d3511b,delivered,2017-04-09 17:41:13,2017-04-09 17:55:19,2017-04-10 14:24:47,2017-04-20 09:08:35,2017-05-10,e64fb393e7b32834bb789ff8bb30750e,5,2017-04-21,2017-04-21 22:02:06,-20,11,On Time
4,8e6bfb81e283fa7e4f11123a3fb894f1,delivered,2018-02-10 10:59:03,2018-02-10 15:48:21,2018-02-15 19:36:14,2018-02-28 16:33:35,2018-03-09,f7c4243c7fe1938f181bec41a392bdeb,5,2018-03-01,2018-03-02 10:26:53,-9,18,On Time
5,b18dcdf73be66366873cd26c5724d1dc,delivered,2018-04-06 22:18:54,2018-04-09 20:10:35,2018-04-11 16:48:35,2018-04-12 17:17:53,2018-05-03,15197aa66ff4d0650b5434f1b46cda19,1,2018-04-13,2018-04-16 00:39:37,-21,6,On Time
6,e48aa0d2dcec3a2e87348811bcfdf22b,delivered,2017-06-30 15:38:46,2017-06-30 15:50:17,2017-07-03 16:22:53,2017-07-15 12:57:24,2017-08-03,07f9bee5d1b850860defd761afa7ff16,5,2017-07-16,2017-07-18 19:30:34,-19,15,On Time
7,c31a859e34e3adac22f376954e19b39d,delivered,2018-08-07 23:12:29,2018-08-07 23:25:10,2018-08-08 14:18:00,2018-08-13 18:08:28,2018-10-10,7c6400515c67679fbee952a7525281ef,5,2018-08-14,2018-08-14 21:36:06,-58,6,On Time
8,9c214ac970e84273583ab523dfafd09b,delivered,2017-05-08 13:35:48,2017-05-08 13:50:15,2017-05-09 14:19:29,2017-05-16 16:49:51,2017-05-30,a3f6f7f6f433de0aefbb97da197c554c,5,2017-05-17,2017-05-18 12:05:37,-14,8,On Time
9,b9bf720beb4ab3728760088589c62129,delivered,2018-05-14 10:29:02,2018-05-15 10:37:47,2018-05-15 13:29:00,2018-05-21 17:52:12,2018-06-06,8670d52e15e00043ae7de4c01cc2fe06,4,2018-05-22,2018-05-23 16:45:47,-16,7,On Time


## 4. Reviews and delivery KPI summary

This section calculates the main reviews and delivery KPIs:
- total reviewed orders
- average review score
- on-time orders
- late orders
- on-time delivery rate
- average delivery lead time
- average delivery delay

In [32]:
delivery_kpi_query = """
WITH delivery_base AS (
    SELECT *
    FROM vw_reviews_delivery_summary
),

delivery_kpis AS (
    SELECT
        COUNT(*) AS total_orders,
        ROUND(AVG(delivery_days)::numeric, 4) AS average_delivery_days,
        ROUND(AVG(estimated_delivery_days)::numeric, 4) AS average_estimated_delivery_days,
        ROUND(AVG(delivery_variance_days)::numeric, 4) AS average_delivery_variance_days,
        ROUND(100.0 * AVG(is_late)::numeric, 4) AS late_delivery_rate,
        ROUND(AVG(late_delay_days)::numeric, 4) AS average_late_delay_days
    FROM delivery_base
),

review_kpis AS (
    SELECT
        ROUND(AVG(review_score)::numeric, 4) AS average_review_score
    FROM olist_order_reviews
    WHERE review_score IS NOT NULL
)

SELECT
    d.total_orders,
    d.average_delivery_days,
    d.average_estimated_delivery_days,
    d.average_delivery_variance_days,
    d.late_delivery_rate,
    d.average_late_delay_days,
    r.average_review_score
FROM delivery_kpis d
CROSS JOIN review_kpis r;
"""

df_delivery_kpis = pd.read_sql(delivery_kpi_query, engine)
df_delivery_kpis

,total_orders,average_delivery_days,average_estimated_delivery_days,average_delivery_variance_days,late_delivery_rate,average_late_delay_days,average_review_score
0,96470,12.4968,24.3727,-11.8759,6.7731,10.6201,4.0864


## 5. Review score distribution
This section shows the distribution of review scores from 1 to 5.

In [33]:
reviews_delivery_summary_query = """
SELECT
    order_id,
    delivery_days,
    estimated_delivery_days,
    delivery_variance_days,
    is_late,
    late_delay_days,
    review_score
FROM vw_reviews_delivery_summary
ORDER BY order_id;
"""

df_reviews_delivery_summary = pd.read_sql(reviews_delivery_summary_query, engine)
df_reviews_delivery_summary

,order_id,delivery_days,estimated_delivery_days,delivery_variance_days,is_late,late_delay_days,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,7,16,-9,0,NaN,5.0
1,00018f77f2f0320c557190d7a144bdd3,16,19,-3,0,NaN,4.0
2,000229ec398224ef6ca0657da4fc703e,8,22,-14,0,NaN,5.0
3,00024acbcdf0a6daa1e931b038114c75,6,12,-6,0,NaN,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,25,41,-16,0,NaN,5.0
...,...,...,...,...,...,...,...
96465,fffc94f6ce00a00581880bf54a75a037,17,25,-8,0,NaN,5.0
96466,fffcd46ef2263f404302a634eb57f7eb,9,18,-9,0,NaN,5.0
96467,fffce4705a9662cd70adb13d4a31832d,5,18,-13,0,NaN,5.0
96468,fffe18544ffabc95dfada21779c9644f,2,11,-9,0,NaN,5.0


## 6. Delivery status distribution
This section summarizes order counts across On Time, Late, and Not Delivered statuses.

In [34]:
delivery_status_query = """
SELECT
    CASE
        WHEN is_late = 1 THEN 'Late'
        ELSE 'On Time / Early'
    END AS delivery_status,
    COUNT(*) AS total_orders,
    ROUND(AVG(delivery_days)::numeric, 4) AS avg_delivery_days,
    ROUND(AVG(delivery_variance_days)::numeric, 4) AS avg_delivery_variance_days,
    ROUND(AVG(review_score)::numeric, 4) AS avg_review_score
FROM vw_reviews_delivery_summary
GROUP BY
    CASE
        WHEN is_late = 1 THEN 'Late'
        ELSE 'On Time / Early'
    END
ORDER BY total_orders DESC;
"""

df_delivery_status = pd.read_sql(delivery_status_query, engine)
df_delivery_status

,delivery_status,total_orders,avg_delivery_days,avg_delivery_variance_days,avg_review_score
0,On Time / Early,89936,10.9411,-13.5103,4.2906
1,Late,6534,33.9105,10.6201,2.2718


## 7. Review score by delivery status
This section helps assess whether late deliveries are associated with lower review scores.

In [35]:
review_by_delivery_query = """
SELECT
    delivery_status,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(AVG(review_score)::numeric, 2) AS avg_review_score
FROM vw_reviews_delivery
WHERE review_score IS NOT NULL
GROUP BY delivery_status
ORDER BY total_orders DESC;
"""

df_review_by_delivery = pd.read_sql(review_by_delivery_query, engine)
df_review_by_delivery

,delivery_status,total_orders,avg_review_score
0,On Time,89448,4.29
1,Late,6382,2.27
2,Not Delivered,2843,1.76


## 8. Monthly delivery performance
This section tracks monthly delivery performance and review quality over time.

In [36]:
monthly_delivery_query = """
SELECT
    DATE_TRUNC('month', order_purchase_timestamp)::date AS order_month,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT CASE WHEN delivery_status = 'On Time' THEN order_id END) AS on_time_orders,
    COUNT(DISTINCT CASE WHEN delivery_status = 'Late' THEN order_id END) AS late_orders,
    ROUND(AVG(review_score)::numeric, 2) AS avg_review_score,
    ROUND(AVG(delivery_delay_days)::numeric, 2) AS avg_delivery_delay_days
FROM vw_reviews_delivery
GROUP BY DATE_TRUNC('month', order_purchase_timestamp)::date
ORDER BY order_month;
"""

df_monthly_delivery = pd.read_sql(monthly_delivery_query, engine)
df_monthly_delivery

,order_month,total_orders,on_time_orders,late_orders,avg_review_score,avg_delivery_delay_days
0,2016-09-01,4,0,1,1.00,36.00
1,2016-10-01,324,268,2,3.57,-36.70
2,2016-12-01,1,1,0,5.00,-22.00
3,2017-01-01,800,728,22,4.07,-27.45
4,2017-02-01,1780,1604,49,4.02,-19.22
5,2017-03-01,2682,2430,116,4.07,-12.34
6,2017-04-01,2404,2152,151,4.04,-12.99
7,2017-05-01,3700,3439,106,4.14,-13.51
8,2017-06-01,3245,3040,95,4.15,-12.66
9,2017-07-01,4026,3764,108,4.17,-12.45


## 9. Delay bucket analysis
This section groups delivered orders into simple delivery-delay buckets.

In [37]:
delay_bucket_query = """
SELECT
    CASE
        WHEN delivery_delay_days IS NULL THEN 'Not Delivered'
        WHEN delivery_delay_days <= 0 THEN 'On Time / Early'
        WHEN delivery_delay_days BETWEEN 1 AND 3 THEN '1-3 Days Late'
        WHEN delivery_delay_days BETWEEN 4 AND 7 THEN '4-7 Days Late'
        ELSE '8+ Days Late'
    END AS delay_bucket,
    COUNT(DISTINCT order_id) AS orders_count,
    ROUND(AVG(review_score)::numeric, 2) AS avg_review_score
FROM vw_reviews_delivery
GROUP BY
    CASE
        WHEN delivery_delay_days IS NULL THEN 'Not Delivered'
        WHEN delivery_delay_days <= 0 THEN 'On Time / Early'
        WHEN delivery_delay_days BETWEEN 1 AND 3 THEN '1-3 Days Late'
        WHEN delivery_delay_days BETWEEN 4 AND 7 THEN '4-7 Days Late'
        ELSE '8+ Days Late'
    END
ORDER BY orders_count DESC;
"""

df_delay_bucket = pd.read_sql(delay_bucket_query, engine)
df_delay_bucket

,delay_bucket,orders_count,avg_review_score
0,On Time / Early,89941,4.29
1,Not Delivered,2965,1.76
2,8+ Days Late,2863,1.70
3,1-3 Days Late,1870,3.29
4,4-7 Days Late,1802,2.10


## 10. Order status vs review score
This section checks how review scores vary by order lifecycle status.

In [38]:
order_status_review_query = """
SELECT
    order_status,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(AVG(review_score)::numeric, 2) AS avg_review_score
FROM vw_reviews_delivery
WHERE review_score IS NOT NULL
GROUP BY order_status
ORDER BY total_orders DESC;
"""

df_order_status_review = pd.read_sql(order_status_review_query, engine)
df_order_status_review

,order_status,total_orders,avg_review_score
0,delivered,95832,4.16
1,shipped,1032,2.01
2,canceled,605,1.81
3,unavailable,595,1.53
4,invoiced,309,1.66
5,processing,295,1.28
6,created,3,2.33
7,approved,2,2.50


## 11. Export reviews and delivery outputs
Save all reviews and delivery analysis outputs into the project `sql/outputs/` folder.

In [39]:
OUTPUT_DIR = Path(r"C:\Users\divya\Retail-Intelligence-Platform\sql\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_delivery_kpis.to_csv(OUTPUT_DIR / "delivery_kpi_summary_sql.csv", index=False)
df_reviews_delivery_summary.to_csv(OUTPUT_DIR / "reviews_delivery_summary_sql.csv", index=False)
df_delivery_status.to_csv(OUTPUT_DIR / "delivery_status_summary_sql.csv", index=False)

# keep optional existing outputs if they already exist in your notebook
try:
    df_review_distribution.to_csv(OUTPUT_DIR / "review_distribution_sql.csv", index=False)
except NameError:
    pass

try:
    df_monthly_delivery_reviews.to_csv(OUTPUT_DIR / "monthly_delivery_reviews_sql.csv", index=False)
except NameError:
    pass

print("Reviews & Delivery SQL outputs exported successfully.")
print(f"Saved to: {OUTPUT_DIR}")

Reviews & Delivery SQL outputs exported successfully.
Saved to: C:\Users\divya\Retail-Intelligence-Platform\sql\outputs


## 12. Summary

This notebook created the SQL reviews and delivery analytics layer for the Retail Intelligence Platform.

### Outputs generated
- `reviews_delivery_summary.csv`
- `review_score_distribution.csv`
- `delivery_status_summary.csv`
- `review_by_delivery_status.csv`
- `monthly_delivery_summary.csv`
- `delivery_delay_bucket_summary.csv`
- `order_status_review_summary.csv`